In [25]:
from pathlib import Path

import pandas as pd
from scipy.stats import chi2_contingency, mannwhitneyu, normaltest, shapiro, wilcoxon
from statsmodels.stats.multitest import multipletests

from cliffs_delta import cliffs_delta


# Dataset overview

In [26]:

prs = pd.read_csv('../datasets/processed/rq1/all_eligible_prs.csv')


## Repositories overview

In [27]:
prs['repo_url'] = prs['pr_url'].str.split('/pull/').str[0]

In [28]:
# dataset overview
print(f"Total repositories: {prs['repo_url'].nunique()}")
print(f"Total PRs: {len(prs)}")
print(f"PRs types count:\n{prs['pr_type'].value_counts()}") 


Total repositories: 101
Total PRs: 522
PRs types count:
pr_type
ai       264
human    258
Name: count, dtype: int64


# Metrics data

In [29]:
metrics = pd.read_csv("../datasets/processed/combined_metrics.csv")
metrics["review_duration"] = metrics["review_duration"] / 3600  # seconds → hours

# Exclude CI/checking-bot activity from the human-review effort counts.
metrics["reviewers"] = metrics["human_reviewers"] + metrics["ai_reviewers"]
metrics["review_comments"] = metrics["human_review_comments"] + metrics["ai_review_comments"]
metrics["comments"] = metrics["human_comments"] + metrics["ai_comments"]
metrics["commits"] = metrics["human_commits"] + metrics["ai_commits"]
metrics["revision"] = metrics["human_revision_commits"] + metrics["ai_revision_commits"]

ai_rows = metrics.loc[metrics["pr_type"] == "ai"].copy()
human_rows = metrics.loc[metrics["pr_type"] == "human"].copy()
metrics_by_type = {"ai": ai_rows, "human": human_rows}

# The 13 continuous measures and one binary outcome reported in Table III.
RQ1_CONTINUOUS_METRICS = [
    "reviewers", "review_comments", "comments", "commits",
    "changed_files", "initial_workflow_files", "revision_workflow_files",
    "review_duration", "length_of_discussion", "length_of_description",
    "code_churn", "workflow_code_churn", "build_failure_rate",
]
RQ1_BINARY_OUTCOMES = ["accepted"]
RQ1_METRICS = RQ1_CONTINUOUS_METRICS + RQ1_BINARY_OUTCOMES
missing_rq1_metrics = [column for column in RQ1_METRICS if column not in metrics.columns]
assert not missing_rq1_metrics, f"Missing RQ1 metrics: {missing_rq1_metrics}"


def format_p_value(value):
    return '<0.001' if value < 0.001 else f'{value:.3f}'


def compute_effect_sizes(metric_names):
    test_rows = []

    for column in metric_names:
        ai_col = ai_rows[column].dropna()
        human_col = human_rows[column].dropna()

        cliff_delta_value, size = cliffs_delta(ai_col, human_col)
        u_statistic, p_value = mannwhitneyu(
            ai_col, human_col, alternative="two-sided"
        )

        test_rows.append(
            {
                "metric": column,
                "statistic": u_statistic,
                "p_value": p_value,
                "p_value_simplified": format_p_value(p_value),
                "cliff_delta": cliff_delta_value,
                "size": size,
            }
        )

    return pd.DataFrame(test_rows)

# Compute acceptance rate
def compute_binary_outcomes(metric_names):
    outcome_rows = []

    for column in metric_names:
        ai_col = ai_rows[column].dropna().astype(int)
        human_col = human_rows[column].dropna().astype(int)
        contingency = [
            [int(ai_col.sum()), int((ai_col == 0).sum())],
            [int(human_col.sum()), int((human_col == 0).sum())],
        ]
        chi2_statistic, p_value, _, _ = chi2_contingency(
            contingency, correction=False
        )
       
        # Calculate the total number of observations in the contingency table
        total_n = sum(sum(row) for row in contingency)
         # Calculate Cramér’s V effect size; for a 2×2 table, V = √(χ² / N)
        cramers_v = (chi2_statistic / total_n) ** 0.5
        size = (
            'negligible' if cramers_v < 0.10 else
            'small' if cramers_v < 0.30 else
            'medium' if cramers_v < 0.50 else 'large'
        )
        outcome_rows.append(
            {
                'metric': column,
                'statistic': chi2_statistic,
                'p_value': p_value,
                'p_value_simplified': format_p_value(p_value),
                'cramers_v': cramers_v,
                'size': size,
            }
        )

    return pd.DataFrame(outcome_rows)




# Metric labels

In [30]:
metric_labels = {
    'reviewers': 'Number of reviewers',
    'review_comments': 'Number of review comments',
    'comments': 'Number of comments',
    'commits': 'Number of commits',
    'changed_files': 'Number of total changed files',
    'initial_workflow_files': 'Number of workflow files change (first commit)',
    'revision_workflow_files': 'Number of workflow files change (revision commits)',
    'review_duration': 'Review duration (hours)',
    'length_of_discussion': 'Length of discussion (characters)',
    'length_of_description': 'Length of description (characters)',
    'code_churn': 'Total code churn',
    'workflow_code_churn': 'Code churn (workflow files)',
    'accepted': 'Acceptance rate',
    'build_failure_rate': 'Build failure rate',
    'workflow_failure_rate': 'Workflow failure rate',
}


def metric_label(metric):
    return metric_labels.get(metric, metric.replace("_", " ").title())


## Shapiro–Wilk normality assessment

The Shapiro–Wilk test is applied separately to APR and HPR observations for each of the 13 continuous RQ1 metrics reported in Table III. The null hypothesis is that a group's observations come from a normal distribution; p < 0.05 indicates evidence against normality. The test is used as a distributional diagnostic supporting the non-parametric Mann–Whitney U analysis, not as the sole basis for selecting a test.

In [31]:
normality_rows = []

for metric in RQ1_CONTINUOUS_METRICS:
    for pr_type, group_name in (("ai", "APR"), ("human", "HPR")):
        values = metrics.loc[metrics["pr_type"].eq(pr_type), metric].dropna().astype(float)

        if len(values) < 3 or values.nunique() < 2:
            w_statistic = float("nan")
            p_value = float("nan")
            normality_rejected = pd.NA
            conclusion = "Not testable"
        else:
            w_statistic, p_value = shapiro(values)
            normality_rejected = bool(p_value < 0.05)
            conclusion = "Reject normality" if normality_rejected else "Normality not rejected"

        normality_rows.append({
            "Metric": metric_label(metric),
            "metric": metric,
            "Group": group_name,
            "pr_type": pr_type,
            "N": len(values),
            "W": w_statistic,
            "p_value": p_value,
            "p_value_display": (
                "" if pd.isna(p_value) else "<0.001" if p_value < 0.001 else f"{p_value:.3f}"
            ),
            "normality_rejected": normality_rejected,
            "Conclusion": conclusion,
        })

normality_results = pd.DataFrame(normality_rows)
normality_output = Path("../datasets/processed/rq1/shapiro_wilk_normality_tests.csv")
normality_output.parent.mkdir(parents=True, exist_ok=True)
normality_results.to_csv(normality_output, index=False)

print(f"Wrote {normality_output} ({len(normality_results)} metric/group tests)")


Wrote ../datasets/processed/rq1/shapiro_wilk_normality_tests.csv (26 metric/group tests)


In [32]:
cliff_p_value = compute_effect_sizes(RQ1_CONTINUOUS_METRICS)
acceptance_statistics = compute_binary_outcomes(RQ1_BINARY_OUTCOMES)


## RQ1 review-effort table (five-number summaries + statistical tests)


In [33]:
# ---------------------------------------------------------------------------
# Full RQ1 review-effort table: five-number summary per PR type for every
# metric, plus the inferential statistics (Mann-Whitney U + Cliff's delta for
# the continuous metrics, Pearson chi-square + Cramer's V for the binary
# acceptance outcome). Written as CSV to datasets/processed/rq1/.
# validate_results.py checks the paper's RQ1 numbers against this file.
# ---------------------------------------------------------------------------
continuous_lookup = cliff_p_value.set_index('metric')
binary_lookup = acceptance_statistics.set_index('metric')

review_effort_rows = []
for column in RQ1_CONTINUOUS_METRICS + RQ1_BINARY_OUTCOMES:
    row = {'Metric': metric_label(column)}

    for group, prefix in (('ai', 'AI'), ('human', 'Human')):
        values = metrics_by_type[group][column].dropna()
        row[f'{prefix}_Min'] = values.min()
        row[f'{prefix}_Q1'] = values.quantile(0.25)
        row[f'{prefix}_Median'] = values.median()
        row[f'{prefix}_Mean'] = values.mean()
        row[f'{prefix}_Q3'] = values.quantile(0.75)
        row[f'{prefix}_Max'] = values.max()

    if column in continuous_lookup.index:
        stats = continuous_lookup.loc[column]
        row['test'] = 'Mann-Whitney U'
        row['cliff_delta'] = stats['cliff_delta']
    else:
        stats = binary_lookup.loc[column]
        row['test'] = 'Pearson chi-square'
        row['cramers_v'] = stats['cramers_v']

    row['statistic'] = stats['statistic']
    row['size'] = stats['size']
    row['p_value'] = stats['p_value']
    # Numeric counterpart of format_p_value: rounded to three decimals and
    # floored at 0.001, so downstream checks can compare numerically.
    # p_value_display carries the "<0.001" form used in the paper's tables.
    row['p_value_simplified'] = max(round(stats['p_value'], 3), 0.001)
    row['p_value_display'] = stats['p_value_simplified']

    review_effort_rows.append(row)

review_effort_statistics = pd.DataFrame(review_effort_rows)

rq1_dir = Path('../datasets/processed/rq1')
rq1_dir.mkdir(parents=True, exist_ok=True)
review_effort_statistics.to_csv(rq1_dir / 'review_effort_statistics.csv', index=False)
print(f"Wrote {rq1_dir / 'review_effort_statistics.csv'} ({len(review_effort_statistics)} metrics)")
display(review_effort_statistics)


Wrote ../datasets/processed/rq1/review_effort_statistics.csv (14 metrics)


,Metric,AI_Min,AI_Q1,AI_Median,AI_Mean,AI_Q3,AI_Max,Human_Min,Human_Q1,Human_Median,...,Human_Q3,Human_Max,test,cliff_delta,statistic,size,p_value,p_value_simplified,p_value_display,cramers_v
0,Number of reviewers,0.000000,0.000000,0.000000,0.575758,1.000000,5.000000,0.000000,0.000000,0.000000,...,1.000000,5.000000,Mann-Whitney U,-0.017427,33462.5000,negligible,6.981526e-01,0.698,0.698,NaN
1,Number of review comments,0.000000,0.000000,0.000000,1.060606,0.000000,72.000000,0.000000,0.000000,0.000000,...,0.000000,43.000000,Mann-Whitney U,0.042562,35505.5000,negligible,2.168135e-01,0.217,0.217,NaN
2,Number of comments,0.000000,0.000000,1.000000,1.219697,1.000000,29.000000,0.000000,0.000000,0.000000,...,0.000000,7.000000,Mann-Whitney U,0.367101,46558.0000,medium,1.488175e-16,0.001,<0.001,NaN
3,Number of commits,0.000000,1.000000,2.000000,4.363636,5.000000,30.000000,0.000000,1.000000,1.000000,...,3.000000,102.000000,Mann-Whitney U,0.142016,38892.5000,negligible,2.881111e-03,0.003,0.003,NaN
4,Number of total changed files,1.000000,1.000000,3.000000,12.655303,8.000000,449.000000,1.000000,1.000000,2.000000,...,7.000000,1354.000000,Mann-Whitney U,0.056466,35979.0000,negligible,2.542102e-01,0.254,0.254,NaN
5,Number of workflow files change (first commit),1.000000,1.000000,1.000000,1.575758,1.000000,14.000000,1.000000,1.000000,1.000000,...,1.000000,9.000000,Mann-Whitney U,0.043208,35527.5000,negligible,2.002571e-01,0.200,0.200,NaN
6,Number of workflow files change (revision comm...,0.000000,1.000000,1.000000,1.648649,2.000000,14.000000,0.000000,0.000000,1.000000,...,2.000000,19.000000,Mann-Whitney U,0.116948,9918.5000,negligible,7.888291e-02,0.079,0.079,NaN
7,Review duration (hours),0.002222,0.161597,1.355000,53.788702,18.504583,1480.319167,0.001667,0.097708,0.710556,...,7.940556,6500.317778,Mann-Whitney U,0.128509,38432.5000,negligible,1.109041e-02,0.011,0.011,NaN
8,Length of discussion (characters),0.000000,178.500000,710.000000,6791.299242,3541.000000,899679.000000,0.000000,0.000000,1188.000000,...,4303.750000,762506.000000,Mann-Whitney U,0.006034,34261.5000,negligible,9.045139e-01,0.905,0.905,NaN
9,Length of description (characters),0.000000,250.250000,438.000000,1054.511364,958.500000,77435.000000,0.000000,2.500000,378.000000,...,1105.250000,5847.000000,Mann-Whitney U,0.148916,39127.5000,small,3.207831e-03,0.003,0.003,NaN


## Holm–Bonferroni correction

In [34]:
reject, p_holm, _, _ = multipletests(review_effort_statistics['p_value'], alpha=0.05, method='holm')
review_effort_statistics['p_value_holm'] = p_holm
review_effort_statistics['significant_holm'] = reject
display(review_effort_statistics[['Metric', 'p_value', 'p_value_holm', 'significant_holm']])

,Metric,p_value,p_value_holm,significant_holm
0,Number of reviewers,6.981526e-01,1.000000e+00,False
1,Number of review comments,2.168135e-01,1.000000e+00,False
2,Number of comments,1.488175e-16,2.083445e-15,True
3,Number of commits,2.881111e-03,3.169222e-02,True
4,Number of total changed files,2.542102e-01,1.000000e+00,False
5,Number of workflow files change (first commit),2.002571e-01,1.000000e+00,False
6,Number of workflow files change (revision comm...,7.888291e-02,5.521804e-01,False
7,Review duration (hours),1.109041e-02,9.981369e-02,False
8,Length of discussion (characters),9.045139e-01,1.000000e+00,False
9,Length of description (characters),3.207831e-03,3.207831e-02,True


## Repository-level analysis (paired)

Following Hora and Robbes (MSR '26), the metrics that remain significant after the Holm–Bonferroni correction are compared within the same repository. For each repository, AI and human PRs are summarised separately (median for continuous metrics, mean for acceptance, i.e., the repository's acceptance rate). The paired values are compared with a Wilcoxon signed-rank test, with Cliff's delta as the effect size. Shapiro–Wilk and D'Agostino tests on the paired differences check normality.

In [ ]:
significant_metrics = [RQ1_METRICS[i] for i in review_effort_statistics.index[review_effort_statistics['significant_holm']]]

# One row per repository, with AI and human values side by side for each metric.
aggregation = {m: 'mean' if m in RQ1_BINARY_OUTCOMES else 'median' for m in significant_metrics}
repo_level = metrics.groupby(['repo_url', 'pr_type']).agg(aggregation).unstack('pr_type')

repo_rows = []
for metric in significant_metrics:
    pairs = repo_level[metric].dropna()
    ai_values, human_values = pairs['ai'], pairs['human']
    differences = ai_values - human_values

    w_statistic, p_value = wilcoxon(ai_values, human_values)
    cliff_delta_value, size = cliffs_delta(ai_values, human_values)

    repo_rows.append({
        'Metric': metric_label(metric),
        'repositories': len(pairs),
        'AI_Median': ai_values.median(),
        'Human_Median': human_values.median(),
        'shapiro_p': shapiro(differences).pvalue,
        'dagostino_p': normaltest(differences).pvalue,
        'W': w_statistic,
        'p_value': p_value,
        'cliff_delta': cliff_delta_value,
        'size': size,
    })

repo_level_statistics = pd.DataFrame(repo_rows)

# The paired tests form a second family; correct them with Holm as well.
reject, p_holm, _, _ = multipletests(repo_level_statistics['p_value'], alpha=0.05, method='holm')
repo_level_statistics['p_value_holm'] = p_holm
repo_level_statistics['significant_holm'] = reject
repo_level_statistics['p_value_display'] = repo_level_statistics['p_value'].map(format_p_value)
repo_level_statistics['p_value_holm_display'] = repo_level_statistics['p_value_holm'].map(format_p_value)
display(repo_level_statistics)